# LeetCode 3928 — Minimum Cost to Buy Apples II bằng Python

- Link: https://leetcode.com/problems/minimum-cost-to-buy-apples-ii/
- Độ khó: **Hard**
- Chủ đề: Đồ thị vô hướng có trọng số, đường đi ngắn nhất (Dijkstra, Floyd–Warshall), đồ thị phân tầng (layered graph), cắt tỉa (pruning) theo cận

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minCost(self, n: int, prices: List[int], roads: List[List[int]]) -> List[int]:
```

## 1. Giới thiệu & mục tiêu học

Có `n` cửa hàng, mỗi cửa hàng bán táo với giá `prices[i]`. Các con đường hai chiều `[u, v, cost, tax]`: đi **tay không** tốn `cost`; đi **mang táo** tốn `cost · tax`. Xuất phát từ cửa hàng `i`, ta có thể mua táo ngay tại `i` (giá `prices[i]`), hoặc **đi tay không** tới cửa hàng `j`, mua táo ở `j` (giá `prices[j]`) rồi **mang táo về `i`**. Đường đi và đường về **có thể khác nhau**. Với mỗi `i`, tìm tổng chi phí nhỏ nhất.

Bài rất đẹp ở chỗ: tổng chi phí qua `j` là `d_empty(i, j) + prices[j] + d_loaded(j, i)`, trong đó **hai khoảng cách là hai đường đi ngắn nhất độc lập** (hai trọng số khác nhau trên cùng một đồ thị). Ta cần `n` lần Dijkstra, mỗi lần cho mỗi trọng số.

Sau notebook này bạn sẽ:

1. Biểu diễn đáp án `ans[i] = min_j ( dist_cost(i, j) + prices[j] + dist_taxed(i, j) )` (vì đồ thị vô hướng, `dist_taxed(j, i) = dist_taxed(i, j)`).
2. Hiểu vì sao **không** gộp thành một Dijkstra duy nhất với trọng số `cost·(1 + tax)` (đường đi và đường về có thể khác nhau).
3. Dựng **đồ thị phân tầng** hai tầng (tay không / mang táo) và chạy Dijkstra từ `(i, tay không)` đến `(i, mang táo)` — cách giải đúng "theo định nghĩa".
4. Cài Dijkstra bằng `heapq`, và **cắt tỉa**: dừng sớm khi không thể cải thiện đáp án hiện có.
5. Viết 4 lời giải (Floyd–Warshall, Dijkstra phân tầng, hai Dijkstra, hai Dijkstra có cắt tỉa) và đối chiếu bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `n`, mảng `prices` độ dài `n`, và `roads[k] = [u, v, cost, tax]` (đường hai chiều). Với mỗi cửa hàng `i`, một trong hai cách:

- mua tại chỗ: `prices[i]`;
- đi tay không tới `j` bằng bất kỳ dãy đường nào (mỗi đường tốn `cost`), mua ở `j` (`prices[j]`), rồi quay về `i` mang táo (mỗi đường tốn `cost·tax`). Đường về có thể khác đường đi.

`ans[i]` = chi phí nhỏ nhất. Trả về mảng `ans`.

**Ví dụ 1:** `n = 2`, `prices = [8,3]`, `roads = [[0,1,1,2]]` → `[6,3]` (từ 0: đi 1, mua 3, về `1·2 = 2` → `6`).

**Ví dụ 2:** `n = 3`, `prices = [9,4,6]`, `roads = [[0,1,1,3],[1,2,4,2]]` → `[8,4,6]`.

**Ví dụ 3:** `n = 3`, `prices = [10,11,1]`, `roads = [[0,2,1,3],[1,2,3,4],[0,1,5,2]]` → `[5,11,1]`.

**Ràng buộc:** `1 ≤ n ≤ 1000`, `1 ≤ prices[i] ≤ 10^9`, `0 ≤ len(roads) ≤ min(n(n−1)/2, 2000)`, `1 ≤ cost ≤ 10^9`, `1 ≤ tax ≤ 100`, không có cạnh lặp.

In [1]:
import heapq, random, time
from itertools import combinations

INF = float("inf")
EXAMPLES = [
    ((2, [8, 3], [[0, 1, 1, 2]]), [6, 3]),
    ((3, [9, 4, 6], [[0, 1, 1, 3], [1, 2, 4, 2]]), [8, 4, 6]),
    ((3, [10, 11, 1], [[0, 2, 1, 3], [1, 2, 3, 4], [0, 1, 5, 2]]), [5, 11, 1]),
]
for args, exp in EXAMPLES:
    print(args, "-> kỳ vọng", exp)

(2, [8, 3], [[0, 1, 1, 2]]) -> kỳ vọng [6, 3]
(3, [9, 4, 6], [[0, 1, 1, 3], [1, 2, 4, 2]]) -> kỳ vọng [8, 4, 6]
(3, [10, 11, 1], [[0, 2, 1, 3], [1, 2, 3, 4], [0, 1, 5, 2]]) -> kỳ vọng [5, 11, 1]


## 3. Kiến thức nền

### 3.1. Danh sách kề với hai trọng số

Mỗi con đường `[u, v, cost, tax]` hai chiều ⇒ thêm `v` vào danh sách của `u` và `u` vào danh sách của `v`, kèm **hai trọng số**: `cost` (tay không) và `cost·tax` (mang táo). Ta lưu bộ ba `(hàng xóm, cost, cost·tax)`.

In [2]:
def build_graph(n, roads):
    adj = [[] for _ in range(n)]
    for u, v, c, t in roads:
        adj[u].append((v, c, c * t))
        adj[v].append((u, c, c * t))
    return adj

adj = build_graph(3, [[0, 2, 1, 3], [1, 2, 3, 4], [0, 1, 5, 2]])
for u in range(3):
    print(u, adj[u])

0 [(2, 1, 3), (1, 5, 10)]
1 [(2, 3, 12), (0, 5, 10)]
2 [(0, 1, 3), (1, 3, 12)]


### 3.2. Dijkstra với `heapq`

Dijkstra tìm khoảng cách ngắn nhất từ một nguồn khi mọi trọng số **không âm**. Ý tưởng: dùng hàng đợi ưu tiên chứa `(khoảng cách tạm thời, nút)`; lấy nút có khoảng cách nhỏ nhất, **nếu bản ghi đã cũ** (`du > dist[u]`) thì bỏ qua; ngược lại thử cải thiện khoảng cách của các hàng xóm.

`weight_idx` chọn trọng số dùng: `1` = `cost` (tay không), `2` = `cost·tax` (mang táo).

In [3]:
def dijkstra(src, adj, weight_idx):
    n = len(adj)
    dist = [INF] * n
    dist[src] = 0
    heap = [(0, src)]
    while heap:
        du, u = heapq.heappop(heap)
        if du > dist[u]:                     # bản ghi cũ trong heap
            continue
        for e in adj[u]:
            v = e[0]
            nd = du + e[weight_idx]
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return dist

adj = build_graph(3, [[0, 2, 1, 3], [1, 2, 3, 4], [0, 1, 5, 2]])
print("tay không từ 1:", dijkstra(1, adj, 1))     # 1->2: 3 ; 1->0: min(5, 3+1) = 4
print("mang táo từ 1:", dijkstra(1, adj, 2))      # 1->2: 12 ; 1->0: min(10, 12+3) = 10

tay không từ 1: [4, 0, 3]
mang táo từ 1: [10, 0, 12]


### 3.3. Đồ thị phân tầng (layered graph)

Để biểu diễn "trước khi mua đi tay không, sau khi mua mang táo", ta nhân đôi mỗi nút thành `(v, 0)` (chưa mua) và `(v, 1)` (đã mua):

- `(u, 0) → (v, 0)` với trọng số `cost` (đi tay không);
- `(v, 0) → (v, 1)` với trọng số `prices[v]` (mua tại `v`);
- `(u, 1) → (v, 1)` với trọng số `cost·tax` (mang táo).

Một đường đi từ `(i, 0)` đến `(i, 1)` chính là "đi tay không tới `j`, mua, mang về `i`", và Dijkstra tìm đường rẻ nhất — **không cần giả định** đường đi và đường về giống nhau.

In [4]:
# Mã hóa nút: (v, phase) -> v + phase * n
n = 3
prices = [10, 11, 1]
print("nút tầng 0:", [(v, 0) for v in range(n)], "-> chỉ số", list(range(n)))
print("nút tầng 1:", [(v, 1) for v in range(n)], "-> chỉ số", [v + n for v in range(n)])
print("cạnh mua tại v: chỉ số v -> v + n với trọng số prices[v]:", [(v, v + n, prices[v]) for v in range(n)])

nút tầng 0: [(0, 0), (1, 0), (2, 0)] -> chỉ số [0, 1, 2]
nút tầng 1: [(0, 1), (1, 1), (2, 1)] -> chỉ số [3, 4, 5]
cạnh mua tại v: chỉ số v -> v + n với trọng số prices[v]: [(0, 3, 10), (1, 4, 11), (2, 5, 1)]


### 3.4. Floyd–Warshall

Ma trận khoảng cách `D[i][j]`; với mỗi nút trung gian `k`: `D[i][j] = min(D[i][j], D[i][k] + D[k][j])`. Độ phức tạp `O(n³)`: tốt để làm **chuẩn đối chiếu** với đồ thị nhỏ.

In [5]:
def floyd(n, roads, which):
    D = [[INF] * n for _ in range(n)]
    for i in range(n):
        D[i][i] = 0
    for u, v, c, t in roads:
        w = c if which == "cost" else c * t
        D[u][v] = min(D[u][v], w)
        D[v][u] = min(D[v][u], w)
    for k in range(n):
        Dk = D[k]
        for i in range(n):
            dik = D[i][k]
            if dik == INF:
                continue
            Di = D[i]
            for j in range(n):
                if dik + Dk[j] < Di[j]:
                    Di[j] = dik + Dk[j]
    return D

roads = [[0, 2, 1, 3], [1, 2, 3, 4], [0, 1, 5, 2]]
print(floyd(3, roads, "cost"))
print(floyd(3, roads, "taxed"))

[[0, 4, 1], [4, 0, 3], [1, 3, 0]]
[[0, 10, 3], [10, 0, 12], [3, 12, 0]]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 3**: `prices = [10, 11, 1]`, `roads = [[0,2,1,3],[1,2,3,4],[0,1,5,2]]`.

Khoảng cách tay không `d(i, j)` (theo `cost`): `d(0,2) = 1`, `d(1,2) = 3` (hoặc `5 + 1 = 6`), `d(0,1) = min(5, 1 + 3) = 4`.

Khoảng cách mang táo `t(i, j)` (theo `cost·tax`): `t(0,2) = 3`, `t(1,2) = 12`, `t(0,1) = min(10, 3 + 12) = 10`.

- **Từ 0:** tại chỗ `10`; qua `2`: `d(0,2) + prices[2] + t(0,2) = 1 + 1 + 3 = 5`; qua `1`: `4 + 11 + 10 = 25`. Min `5` ✓.
- **Từ 1:** tại chỗ `11`; qua `2`: `3 + 1 + 12 = 16`; qua `0`: `4 + 10 + 10 = 24`. Min `11` ✓.
- **Từ 2:** tại chỗ `1` ✓.

Công thức: `ans[i] = min( prices[i], min_j ( d(i, j) + prices[j] + t(i, j) ) )`, trong đó `j = i` cho đúng `prices[i]` (`d = t = 0`).

In [6]:
roads = [[0, 2, 1, 3], [1, 2, 3, 4], [0, 1, 5, 2]]
prices = [10, 11, 1]
D = floyd(3, roads, "cost")
T = floyd(3, roads, "taxed")
for i in range(3):
    cands = {j: D[i][j] + prices[j] + T[j][i] for j in range(3)}
    print(f"i={i}: các ứng viên theo j = {cands} -> ans = {min(cands.values())}")

i=0: các ứng viên theo j = {0: 10, 1: 25, 2: 5} -> ans = 5
i=1: các ứng viên theo j = {0: 24, 1: 11, 2: 16} -> ans = 11
i=2: các ứng viên theo j = {0: 14, 1: 26, 2: 1} -> ans = 1


## 5. Cách 1 — Brute Force: Floyd–Warshall hai ma trận

**Ý tưởng.** Tính `D` (theo `cost`) và `T` (theo `cost·tax`) bằng Floyd–Warshall, rồi với mỗi `i`: `ans[i] = min_j D[i][j] + prices[j] + T[j][i]` (bỏ qua `j` không tới được).

In [7]:
def with_floyd(n, prices, roads):
    D = floyd(n, roads, "cost")
    T = floyd(n, roads, "taxed")
    ans = []
    for i in range(n):
        best = INF
        for j in range(n):
            if D[i][j] < INF:
                best = min(best, D[i][j] + prices[j] + T[j][i])
        ans.append(best)
    return ans

for args, exp in EXAMPLES:
    got = with_floyd(*args)
    print(args[0], "->", got)
    assert got == exp

2 -> [6, 3]
3 -> [8, 4, 6]
3 -> [5, 11, 1]


**Độ phức tạp.** `O(n³)`: với `n = 1000` là `10^9` phép ⇒ không khả thi trong Python. **Không gian** `O(n²)`.

**Nút thắt.** Ta chỉ có tối đa 2000 cạnh — đồ thị **thưa** — nên Dijkstra từ mỗi nguồn (`O(m log n)`) rẻ hơn nhiều so với Floyd.

## 6. Bước đột phá

**Quan sát 1 — hai bài đường đi ngắn nhất độc lập.** Với `i` cố định, `ans[i]` là min của một tổng hai thành phần **độc lập**: `d(i, j)` chỉ phụ thuộc vào trọng số `cost`, `t(j, i)` chỉ phụ thuộc vào trọng số `cost·tax`. Nên ta chạy Dijkstra từ `i` **hai lần** (một lần mỗi trọng số) rồi quét `j`.

**Quan sát 2 — vì sao không gộp làm một Dijkstra?** Nếu dùng trọng số `cost·(1 + tax)` ta ép đường đi và đường về **trùng nhau** (đi và về cùng một đường). Đề cho phép khác nhau nên có thể rẻ hơn (mục 13 cho phản ví dụ).

**Quan sát 3 — một cách giải đúng "tự động".** Đồ thị phân tầng (mục 3.3) mã hóa đúng định nghĩa; Dijkstra từ `(i, 0)` tới `(i, 1)` cho `ans[i]` (và là cách giải độc lập để đối chiếu).

**Quan sát 4 — cắt tỉa.** Vì `tax ≥ 1` nên `t(i, j) ≥ d(i, j)`. Do đó tổng qua `j` là `d + t + prices[j] ≥ 2·d(i, j) + minPrice`. Nếu `2·d(i, j) + minPrice ≥ best` hiện tại thì `j` (và mọi nút xa hơn trong Dijkstra theo `cost`) **không thể cải thiện** ⇒ dừng. Tương tự cho Dijkstra theo `cost·tax`: dừng khi khoảng cách `≥ best − minPrice`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Dijkstra trên đồ thị phân tầng, mỗi nguồn một lần

**Giải thích từng đoạn.**

1. Tầng 0: chỉ số `v`, tầng 1: chỉ số `v + n`.
2. Với mỗi nguồn `i`: Dijkstra bắt đầu `(i, 0)`; khi lấy `(v, 0)` ra, thử cạnh mua `(v, 1)` với trọng số `prices[v]`; ở tầng 0 dùng `cost`, ở tầng 1 dùng `cost·tax`.
3. `ans[i] = dist[(i, 1)]`.

In [8]:
def layered_dijkstra(n, prices, roads):
    adj = build_graph(n, roads)
    ans = []
    for src in range(n):
        dist = [INF] * (2 * n)
        dist[src] = 0
        heap = [(0, src)]
        while heap:
            du, u = heapq.heappop(heap)
            if du > dist[u]:
                continue
            if u < n:                                 # tầng 0: chưa mua
                nd = du + prices[u]                    # mua tại u -> sang tầng 1
                if nd < dist[u + n]:
                    dist[u + n] = nd
                    heapq.heappush(heap, (nd, u + n))
                for v, c, tc in adj[u]:
                    nd = du + c
                    if nd < dist[v]:
                        dist[v] = nd
                        heapq.heappush(heap, (nd, v))
            else:                                      # tầng 1: đã mua, mang táo
                w = u - n
                for v, c, tc in adj[w]:
                    nd = du + tc
                    if nd < dist[v + n]:
                        dist[v + n] = nd
                        heapq.heappush(heap, (nd, v + n))
        ans.append(dist[src + n])
    return ans

for args, exp in EXAMPLES:
    got = layered_dijkstra(*args)
    print(args[0], "->", got)
    assert got == exp

2 -> [6, 3]
3 -> [8, 4, 6]
3 -> [5, 11, 1]


**Độ phức tạp:** `n` lần Dijkstra trên đồ thị `2n` nút, `2m + n` cạnh ⇒ `O(n · (m + n) log n)`. Với `n = 1000`, `m = 2000`: cỡ `10^7` thao tác heap — vài giây trong Python.

### 7.2. Cách 3 — Hai Dijkstra thường rồi quét `j`

**Giải thích từng đoạn.** Với mỗi nguồn `i`: `d1 = dijkstra(i, adj, 1)` (tay không), `d2 = dijkstra(i, adj, 2)` (mang táo). `ans[i] = min_j (d1[j] + prices[j] + d2[j])` trên các `j` tới được (`d1[j] < ∞`; `d2[j]` cũng hữu hạn vì cùng liên thông).

Cách này đơn giản hơn Cách 2 (hai Dijkstra trên đồ thị `n` nút) và đây là lời giải ta nộp.

In [9]:
def two_dijkstras(n, prices, roads):
    adj = build_graph(n, roads)
    ans = []
    for i in range(n):
        d1 = dijkstra(i, adj, 1)
        d2 = dijkstra(i, adj, 2)
        best = INF
        for j in range(n):
            if d1[j] < INF:
                best = min(best, d1[j] + prices[j] + d2[j])
        ans.append(best)
    return ans

for args, exp in EXAMPLES:
    got = two_dijkstras(*args)
    print(args[0], "->", got)
    assert got == exp

2 -> [6, 3]
3 -> [8, 4, 6]
3 -> [5, 11, 1]


**Độ phức tạp:** `O(n · m log n)` thời gian (hai Dijkstra mỗi nguồn), `O(n + m)` không gian.

### 7.3. Cách 4 — Hai Dijkstra có cắt tỉa

**Giải thích từng đoạn.**

1. `minP = min(prices)`; `best = prices[i]` ban đầu (mua tại chỗ).
2. **Dijkstra 1 (theo `cost`) có cận:** khi lấy nút `u` có khoảng cách `du` mà `2·du + minP ≥ best` thì dừng (mọi nút còn lại xa hơn). Lưu `d1[u]` cho các nút đã cố định.
3. **Dijkstra 2 (theo `cost·tax`) có cận:** dừng khi `du ≥ best − minP`. Mỗi khi cố định `u` mà `u` đã có `d1[u]` thì thử `d1[u] + du + prices[u]` để cập nhật `best` (và thu hẹp cận `lim = best − minP`).
4. `ans[i] = best`.

Tính đúng: một ứng viên `j` cải thiện `best` chỉ khi `d1 + d2 + prices[j] < best`; vì `d2 ≥ d1` suy ra `d1 < (best − minP)/2` (nên `j` được cố định trong Dijkstra 1) và `d2 < best − minP` (nên được cố định trong Dijkstra 2).

In [10]:
def pruned_dijkstras(n, prices, roads, stats=None):
    adj = build_graph(n, roads)
    minP = min(prices)
    ans = []
    for i in range(n):
        best = prices[i]
        # --- Dijkstra 1: theo cost, dừng khi 2*d + minP >= best ---
        d1 = {}
        seen1 = {i: 0}
        heap = [(0, i)]
        while heap:
            du, u = heapq.heappop(heap)
            if du > seen1.get(u, INF):
                continue
            if 2 * du + minP >= best:
                break
            d1[u] = du
            for v, c, tc in adj[u]:
                nd = du + c
                if nd < seen1.get(v, INF):
                    seen1[v] = nd
                    heapq.heappush(heap, (nd, v))
        # --- Dijkstra 2: theo cost*tax, dừng khi d >= best - minP ---
        lim = best - minP
        seen2 = {i: 0}
        heap = [(0, i)]
        settled2 = 0
        while heap:
            du, u = heapq.heappop(heap)
            if du > seen2.get(u, INF):
                continue
            if du >= lim:
                break
            settled2 += 1
            if u in d1:
                tot = d1[u] + du + prices[u]
                if tot < best:
                    best = tot
                    lim = best - minP
            for v, c, tc in adj[u]:
                nd = du + tc
                if nd < seen2.get(v, INF):
                    seen2[v] = nd
                    heapq.heappush(heap, (nd, v))
        if stats is not None:
            stats.append((len(d1), settled2))
        ans.append(best)
    return ans

for args, exp in EXAMPLES:
    got = pruned_dijkstras(*args)
    print(args[0], "->", got)
    assert got == exp

2 -> [6, 3]
3 -> [8, 4, 6]
3 -> [5, 11, 1]


**Độ phức tạp:** trường hợp xấu nhất vẫn `O(n · m log n)` (khi `prices` lớn so với các khoảng cách nên không cắt được), nhưng thực tế thường giảm mạnh khi `prices` gần nhau hoặc khoảng cách lớn (ví dụ `cost` cỡ `10^9`): xem số nút được cố định ở mục 12.

## 8. TRACE từng bước

Trace **Cách 3** trên Ví dụ 3 (`n = 3`): với mỗi nguồn in `d1`, `d2` và các ứng viên `d1[j] + prices[j] + d2[j]`.

In [11]:
def trace_two_dijkstras(n, prices, roads):
    adj = build_graph(n, roads)
    ans = []
    for i in range(n):
        d1 = dijkstra(i, adj, 1)
        d2 = dijkstra(i, adj, 2)
        print(f"nguồn i = {i}  (mua tại chỗ: {prices[i]})")
        print(f"   d1 (tay không)  = {d1}")
        print(f"   d2 (mang táo)   = {d2}")
        best = INF
        for j in range(n):
            if d1[j] < INF:
                cand = d1[j] + prices[j] + d2[j]
                print(f"   j={j}: d1={d1[j]} + prices={prices[j]} + d2={d2[j]} = {cand}")
                best = min(best, cand)
        print(f"   ans[{i}] = {best}")
        ans.append(best)
    return ans

assert trace_two_dijkstras(*EXAMPLES[2][0]) == [5, 11, 1]

nguồn i = 0  (mua tại chỗ: 10)
   d1 (tay không)  = [0, 4, 1]
   d2 (mang táo)   = [0, 10, 3]
   j=0: d1=0 + prices=10 + d2=0 = 10
   j=1: d1=4 + prices=11 + d2=10 = 25
   j=2: d1=1 + prices=1 + d2=3 = 5
   ans[0] = 5
nguồn i = 1  (mua tại chỗ: 11)
   d1 (tay không)  = [4, 0, 3]
   d2 (mang táo)   = [10, 0, 12]
   j=0: d1=4 + prices=10 + d2=10 = 24
   j=1: d1=0 + prices=11 + d2=0 = 11
   j=2: d1=3 + prices=1 + d2=12 = 16
   ans[1] = 11
nguồn i = 2  (mua tại chỗ: 1)
   d1 (tay không)  = [1, 3, 0]
   d2 (mang táo)   = [3, 12, 0]
   j=0: d1=1 + prices=10 + d2=3 = 14
   j=1: d1=3 + prices=11 + d2=12 = 26
   j=2: d1=0 + prices=1 + d2=0 = 1
   ans[2] = 1


In [12]:
# Trace Dijkstra bên trong: các bước lấy nút khỏi heap (nguồn 1, trọng số cost)
def trace_dijkstra(src, adj, weight_idx):
    n = len(adj)
    dist = [INF] * n
    dist[src] = 0
    heap = [(0, src)]
    step = 0
    while heap:
        du, u = heapq.heappop(heap)
        step += 1
        if du > dist[u]:
            print(f"bước {step}: lấy ({du}, {u}) -> bản ghi cũ, bỏ qua")
            continue
        print(f"bước {step}: lấy ({du}, {u}) -> cố định dist[{u}] = {du}")
        for e in adj[u]:
            v, nd = e[0], du + e[weight_idx]
            if nd < dist[v]:
                print(f"          cải thiện dist[{v}]: {dist[v]} -> {nd}")
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return dist

adj = build_graph(3, [[0, 2, 1, 3], [1, 2, 3, 4], [0, 1, 5, 2]])
assert trace_dijkstra(1, adj, 1) == [4, 0, 3]

bước 1: lấy (0, 1) -> cố định dist[1] = 0
          cải thiện dist[2]: inf -> 3
          cải thiện dist[0]: inf -> 5
bước 2: lấy (3, 2) -> cố định dist[2] = 3
          cải thiện dist[0]: 5 -> 4
bước 3: lấy (4, 0) -> cố định dist[0] = 4
bước 4: lấy (5, 0) -> bản ghi cũ, bỏ qua


## 9. Code nộp LeetCode

Nộp **hai Dijkstra** (Cách 3): ngắn, dễ kiểm tra, đủ nhanh cho `n = 1000`, `m = 2000`.

In [13]:
from typing import List
import heapq

class Solution:
    def minCost(self, n: int, prices: List[int], roads: List[List[int]]) -> List[int]:
        adj = [[] for _ in range(n)]
        for u, v, c, t in roads:
            adj[u].append((v, c, c * t))
            adj[v].append((u, c, c * t))

        INF = float("inf")

        def dijkstra(src, w):
            dist = [INF] * n
            dist[src] = 0
            heap = [(0, src)]
            while heap:
                du, u = heapq.heappop(heap)
                if du > dist[u]:
                    continue
                for e in adj[u]:
                    v = e[0]
                    nd = du + e[w]
                    if nd < dist[v]:
                        dist[v] = nd
                        heapq.heappush(heap, (nd, v))
            return dist

        ans = []
        for i in range(n):
            d1 = dijkstra(i, 1)          # đi tay không: trọng số cost
            d2 = dijkstra(i, 2)          # mang táo về: trọng số cost * tax
            best = prices[i]
            for j in range(n):
                if d1[j] < INF:
                    cand = d1[j] + prices[j] + d2[j]
                    if cand < best:
                        best = cand
            ans.append(best)
        return ans

sol = Solution()
for args, exp in EXAMPLES:
    got = sol.minCost(*args)
    print(args[0], "->", got)
    assert got == exp

2 -> [6, 3]
3 -> [8, 4, 6]
3 -> [5, 11, 1]


## 10. Vì sao thuật toán đúng

**Mô hình.** Một phương án cho cửa hàng `i` là: một đường đi tay không `P` từ `i` tới `j`, mua ở `j`, một đường đi mang táo `Q` từ `j` về `i`. Chi phí `= cost(P) + prices[j] + taxedcost(Q)`. (Trường hợp mua tại chỗ ứng với `j = i`, `P`, `Q` rỗng.)

**Độc lập.** Với `j` cố định, `P` và `Q` được chọn độc lập ⇒ chi phí nhỏ nhất của `j` là `d_cost(i, j) + prices[j] + d_taxed(j, i)`. Đồ thị vô hướng nên `d_taxed(j, i) = d_taxed(i, j)`. Lấy min qua `j` (kể cả `j = i`) và `prices[i]` đã nằm trong `j = i`.

**Dijkstra đúng** vì mọi trọng số dương (`cost ≥ 1`, `cost·tax ≥ 1`).

**Cắt tỉa.** `d_taxed ≥ d_cost` vì mọi đường đi có trọng số mang táo `≥` trọng số tay không (mỗi cạnh `cost·tax ≥ cost`). Nếu một `j` cải thiện được `best` thì `d_cost + d_taxed + prices[j] < best` ⇒ `2·d_cost + minP < best` (do `d_taxed ≥ d_cost`, `prices[j] ≥ minP`) và `d_taxed < best − minP`. Mỗi điều kiện này tương thích với việc Dijkstra cố định nút theo thứ tự khoảng cách tăng dần và dừng khi vượt cận. Giá trị `best` chỉ giảm nên các cận chỉ chặt hơn; dùng `best` tại thời điểm dừng vẫn đúng.

**Đồ thị phân tầng** mã hóa trực tiếp định nghĩa nên cho cùng đáp án (kiểm chứng ở mục 12).

## 11. Phân tích Time/Space

Đặt `n` là số cửa hàng, `m = len(roads) ≤ 2000`.

| Cách | Thời gian | Không gian |
|---|---|---|
| Floyd–Warshall | `O(n³)` | `O(n²)` |
| Dijkstra phân tầng | `O(n·(n + m) log n)` | `O(n + m)` |
| Hai Dijkstra | `O(n·m log n)` | `O(n + m)` |
| Hai Dijkstra + cắt tỉa | `O(n·m log n)` xấu nhất; thường nhỏ hơn | `O(n + m)` |

Với `n = 1000`, `m = 2000`: mỗi Dijkstra ~ `2m = 4000` lần xét cạnh và `≤ 4000` thao tác heap ⇒ cỡ `2000 · 8000 ≈ 1.6·10^7` thao tác heap/cạnh tổng cộng: khoảng 2–3 giây trong Python (xem mục 12).

Giá trị lớn nhất của đáp án: một đường đi dài `≤ n − 1 = 999` cạnh, mỗi cạnh `≤ 10^9 · 100`: tổng `≤ 10^14` — Python không lo tràn (ngôn ngữ khác cần `long long`).

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, không có đường, đồ thị không liên thông, `tax = 1`), đối chiếu ngẫu nhiên giữa 5 cách có seed, và thời gian ở `n = 1000`.

In [14]:
sol = Solution()
for args, exp in EXAMPLES:
    assert sol.minCost(*args) == exp

assert sol.minCost(1, [5], []) == [5]                                   # n = 1, không có đường
assert sol.minCost(3, [4, 9, 7], []) == [4, 9, 7]                        # không có đường nào
assert sol.minCost(4, [10, 3, 8, 2], [[0, 1, 1, 1], [2, 3, 1, 100]]) == [5, 3, 8, 2]   # hai thành phần liên thông riêng
assert sol.minCost(2, [8, 3], [[0, 1, 1, 1]]) == [5, 3]                  # tax = 1: đi 1 + mua 3 + về 1
print("ví dụ + edge case OK")

ví dụ + edge case OK


In [15]:
def gen_case(rng, nmax, price_hi, cost_hi, tax_hi, edge_factor=2):
    n = rng.randint(1, nmax)
    max_edges = n * (n - 1) // 2
    m = rng.randint(0, min(max_edges, edge_factor * n))
    edges = set()
    while len(edges) < m:
        u, v = rng.sample(range(n), 2)
        if u > v:
            u, v = v, u
        edges.add((u, v))
    roads = [[u, v, rng.randint(1, cost_hi), rng.randint(1, tax_hi)] for u, v in sorted(edges)]
    prices = [rng.randint(1, price_hi) for _ in range(n)]
    return n, prices, roads

rng = random.Random(3928)
for _ in range(300):
    args = gen_case(rng, 9, rng.choice([5, 60, 10**9]), rng.choice([3, 20, 10**9]), rng.choice([1, 5, 100]))
    exp = with_floyd(*args)
    assert layered_dijkstra(*args) == exp, args
    assert two_dijkstras(*args) == exp, args
    assert pruned_dijkstras(*args) == exp, args
    assert sol.minCost(*args) == exp, args
print("300 test nhỏ: 5 cách khớp nhau")

for _ in range(40):
    args = gen_case(rng, 60, rng.choice([20, 1000, 10**9]), rng.choice([5, 100, 10**9]), 100, edge_factor=3)
    exp = with_floyd(*args)
    assert layered_dijkstra(*args) == pruned_dijkstras(*args) == sol.minCost(*args) == exp
print("40 test vừa khớp")

300 test nhỏ: 5 cách khớp nhau


40 test vừa khớp


In [16]:
# Thời gian ở giới hạn n = 1000, m = 2000
rng = random.Random(1)
n = 1000
edges = set()
while len(edges) < 2000:
    u, v = rng.sample(range(n), 2)
    edges.add((min(u, v), max(u, v)))
edges = sorted(edges)
scenarios = {
    "cost nhỏ, giá lớn (khó cắt tỉa)": ((1, 10), (10**8, 10**9)),
    "cost lớn, giá lớn": ((1, 10**9), (1, 10**9)),
    "mọi thứ nhỏ": ((1, 10), (1, 100)),
}
for name, (cr, pr) in scenarios.items():
    roads = [[u, v, rng.randint(*cr), rng.randint(1, 100)] for u, v in edges]
    prices = [rng.randint(*pr) for _ in range(n)]
    t0 = time.time(); a = sol.minCost(n, prices, roads); t1 = time.time() - t0
    stats = []
    t0 = time.time(); b = pruned_dijkstras(n, prices, roads, stats); t2 = time.time() - t0
    assert a == b
    avg1 = sum(s[0] for s in stats) / n; avg2 = sum(s[1] for s in stats) / n
    print(f"{name:34s}: hai Dijkstra {t1:.2f}s | cắt tỉa {t2:.2f}s | trung bình nút cố định D1 = {avg1:.0f}, D2 = {avg2:.0f} / {n}")

cost nhỏ, giá lớn (khó cắt tỉa)   : hai Dijkstra 1.68s | cắt tỉa 1.79s | trung bình nút cố định D1 = 971, D2 = 536 / 1000


cost lớn, giá lớn                 : hai Dijkstra 2.10s | cắt tỉa 0.01s | trung bình nút cố định D1 = 3, D2 = 1 / 1000


mọi thứ nhỏ                       : hai Dijkstra 1.77s | cắt tỉa 0.59s | trung bình nút cố định D1 = 549, D2 = 2 / 1000


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Gộp thành một Dijkstra với trọng số `cost·(1 + tax)`

**Code sai:** `ans[i] = min_j (dist_{c(1+t)}(i, j) + prices[j])`.
**Vì sao sai:** nó ép đường đi và đường về là **cùng một đường**. Đề cho phép đường đi và đường về khác nhau; đường tay không ngắn nhất có thể có tax lớn, còn đường mang táo rẻ nhất lại là đường khác.
**Code đúng:** hai Dijkstra riêng cho hai trọng số.

In [17]:
def wrong_single_weight(n, prices, roads):
    adj = [[] for _ in range(n)]
    for u, v, c, t in roads:
        adj[u].append((v, c, c * (1 + t)))          # SAI: một trọng số duy nhất c*(1+t) cho cả đi lẫn về
        adj[v].append((u, c, c * (1 + t)))
    ans = []
    for i in range(n):
        d = dijkstra(i, adj, 2)
        ans.append(min([prices[i]] + [d[j] + prices[j] for j in range(n) if d[j] < INF]))
    return ans

rng = random.Random(7)
found = None
for _ in range(5000):
    args = gen_case(rng, 6, 30, 10, 20)
    if wrong_single_weight(*args) != with_floyd(*args):
        found = args
        break
print("phản ví dụ:", found)
print("sai :", wrong_single_weight(*found), " đúng:", with_floyd(*found))

phản ví dụ: (5, [4, 15, 25, 19, 27], [[0, 1, 3, 11], [0, 2, 10, 1], [0, 3, 6, 9], [0, 4, 9, 20], [1, 2, 1, 4], [1, 3, 1, 7], [1, 4, 10, 16], [2, 3, 10, 19], [2, 4, 4, 9], [3, 4, 5, 14]])
sai : [4, 15, 20, 19, 27]  đúng: [4, 15, 18, 19, 27]


### Lỗi 2 — Áp dụng `tax` cho cả đường đi

**Code sai:** nhân `tax` ở cả chiều đi (`d` theo `cost·tax` hai lần).
**Vì sao sai:** chỉ chuyến **về mang táo** mới chịu `tax`; Ví dụ 1 cho `6` chứ không phải `1·2 + 3 + 1·2 = 7`.
**Code đúng:** đi theo `cost`, về theo `cost·tax`.

In [18]:
def wrong_tax_both_ways(n, prices, roads):
    adj = build_graph(n, roads)
    ans = []
    for i in range(n):
        d2 = dijkstra(i, adj, 2)                 # SAI: dùng d2 cả hai chiều
        ans.append(min(prices[i], min(d2[j] + prices[j] + d2[j] for j in range(n) if d2[j] < INF)))
    return ans

print("sai :", wrong_tax_both_ways(*EXAMPLES[0][0]), " đúng:", Solution().minCost(*EXAMPLES[0][0]))

sai : [7, 3]  đúng: [6, 3]


### Lỗi 3 — Quên các nút không tới được (giá trị `∞`) hoặc quên `j = i`

**Code sai:** `ans[i] = min(d1[j] + prices[j] + d2[j] for j != i)`.
**Vì sao sai:** nút cô lập (hoặc `n = 1`) cho `min` của tập rỗng/vô cực; hơn nữa `j = i` chính là mua tại chỗ.
**Code đúng:** khởi tạo `best = prices[i]` và chỉ xét `j` có `d1[j] < ∞`.

In [19]:
def wrong_skip_self(n, prices, roads):
    adj = build_graph(n, roads)
    ans = []
    for i in range(n):
        d1 = dijkstra(i, adj, 1); d2 = dijkstra(i, adj, 2)
        cands = [d1[j] + prices[j] + d2[j] for j in range(n) if j != i and d1[j] < INF]
        ans.append(min(cands))                    # SAI: lỗi khi không có ứng viên; thiếu prices[i]
    return ans

try:
    wrong_skip_self(2, [8, 3], [])
except ValueError as e:
    print("ValueError:", e)
print("ví dụ 1:", wrong_skip_self(*EXAMPLES[0][0]), " đúng:", Solution().minCost(*EXAMPLES[0][0]), "(nút 1 bị ép phải đi mua chỗ khác: 11 thay vì mua tại chỗ 3)")

ValueError: min() iterable argument is empty
ví dụ 1: [6, 11]  đúng: [6, 3] (nút 1 bị ép phải đi mua chỗ khác: 11 thay vì mua tại chỗ 3)


### Lỗi 4 — Không bỏ qua bản ghi cũ trong heap

Bỏ dòng `if du > dist[u]: continue` vẫn cho kết quả đúng nhưng mỗi nút bị mở rộng nhiều lần (`O(m)` lần thay vì một) ⇒ chậm đáng kể. Đếm số lần mở rộng ở cell dưới.

In [20]:
def dijkstra_count(src, adj, w, skip_stale):
    dist = [INF] * len(adj); dist[src] = 0; heap = [(0, src)]; expanded = 0
    while heap:
        du, u = heapq.heappop(heap)
        if skip_stale and du > dist[u]:
            continue
        expanded += 1
        for e in adj[u]:
            nd = du + e[w]
            if nd < dist[e[0]]:
                dist[e[0]] = nd
                heapq.heappush(heap, (nd, e[0]))
    return expanded

def gen_connected(rng, n, m, price_hi, cost_hi, tax_hi):
    edges = set()
    for v in range(1, n):                                   # cây khung ngẫu nhiên đảm bảo liên thông
        edges.add((rng.randrange(v), v))
    while len(edges) < m:
        u, v = rng.sample(range(n), 2)
        edges.add((min(u, v), max(u, v)))
    roads = [[u, v, rng.randint(1, cost_hi), rng.randint(1, tax_hi)] for u, v in sorted(edges)]
    return n, [rng.randint(1, price_hi) for _ in range(n)], roads

rng = random.Random(2)
n_, prices_, roads_ = gen_connected(rng, 500, 1500, 100, 100, 100)
adj_ = build_graph(n_, roads_)
print("n =", n_, ", m =", len(roads_),
      "| số lần mở rộng nút: có bỏ bản ghi cũ =", dijkstra_count(0, adj_, 1, True),
      ", không bỏ =", dijkstra_count(0, adj_, 1, False))

n = 500 , m = 1500 | số lần mở rộng nút: có bỏ bản ghi cũ = 500 , không bỏ = 781


### Lỗi 5 — Floyd–Warshall `O(n³)` cho `n = 1000`

`10^9` phép là quá nhiều cho Python (và nặng cả với ngôn ngữ biên dịch nếu không tối ưu). Đồ thị chỉ có `≤ 2000` cạnh ⇒ Dijkstra từ mỗi nguồn. So sánh thời gian ở `n = 120`.

In [21]:
rng = random.Random(3)
args = gen_connected(rng, 150, 450, 100, 100, 100)
t0 = time.time(); a = with_floyd(*args); t1 = time.time() - t0
t0 = time.time(); b = Solution().minCost(*args); t2 = time.time() - t0
assert a == b
print(f"n=150, m=450: Floyd {t1:.3f}s | hai Dijkstra {t2:.3f}s -> Floyd tăng theo n^3 (n=1000 gấp ~300 lần)")

n=150, m=450: Floyd 0.249s | hai Dijkstra 0.040s -> Floyd tăng theo n^3 (n=1000 gấp ~300 lần)


## 14. Mẹo & nhận dạng pattern

- **"Đi tới `j` rồi quay về `i` với hai chế độ chi phí khác nhau"** → chia thành **hai bài đường đi ngắn nhất độc lập** cộng với chi phí tại `j`; nhớ đồ thị vô hướng nên hướng về = hướng đi đảo.
- **Chuyển đổi chế độ (tay không → mang táo)** → **đồ thị phân tầng**; luôn dùng được để kiểm chứng và là mô hình tổng quát khi có nhiều chế độ.
- **Đồ thị thưa (`m ≈ 2n`) + `n ≤ 1000`** → Dijkstra từ mọi nguồn (`n · m log n`) tốt hơn Floyd (`n³`).
- **Cắt tỉa theo cận:** nếu một thành phần của tổng luôn `≥` thành phần kia (ở đây `taxed ≥ cost`), có thể bỏ sớm các nút không thể cải thiện đáp án hiện tại.
- Trọng số lớn (`10^9 · 100` và đường dài): dùng số nguyên Python; ngôn ngữ khác cần 64-bit.
- Luôn kiểm tra: đồ thị **có thể không liên thông**; khởi tạo đáp án bằng `prices[i]`.

## 15. Biến thể

1. **Tax chỉ áp dụng cho đường đi** (mang táo đi, tay không về): đổi vai trò hai trọng số — đối xứng.
2. **Mua ở nhiều cửa hàng / nhiều loại hàng:** cần nhiều tầng trong đồ thị phân tầng.
3. **Đồ thị có hướng:** `d_taxed(j, i)` khác `d_taxed(i, j)` nên cần Dijkstra trên **đồ thị đảo chiều** từ `i`.
4. **Có phí cố định mỗi lần mua / số lần dừng tối đa:** thêm chiều trạng thái (số lần mua) → tầng nhiều hơn.
5. **Phiên bản I (đề dễ hơn):** có thể chỉ có đường đi một chiều tính tax — dùng đúng khung `d1 + prices + d2`.

In [22]:
# Biến thể 3: đồ thị CÓ HƯỚNG — d_taxed(j, i) phải tính trên đồ thị đảo chiều
def directed_min_cost(n, prices, arcs):
    # arcs: (u, v, cost, tax) nghĩa là đường một chiều u -> v
    fwd = [[] for _ in range(n)]; rev = [[] for _ in range(n)]
    for u, v, c, t in arcs:
        fwd[u].append((v, c, c * t))
        rev[v].append((u, c, c * t))          # đảo chiều để tính d_taxed(j, i) từ i
    ans = []
    for i in range(n):
        d1 = dijkstra(i, fwd, 1)              # i -> j tay không
        d2 = dijkstra(i, rev, 2)              # j -> i mang táo = i -> j trên đồ thị đảo
        ans.append(min(prices[i], min([d1[j] + prices[j] + d2[j] for j in range(n) if d1[j] < INF and d2[j] < INF], default=INF)))
    return ans

# kiểm chứng bằng đồ thị phân tầng có hướng
def directed_layered(n, prices, arcs):
    fwd = [[] for _ in range(n)]
    for u, v, c, t in arcs:
        fwd[u].append((v, c, c * t))
    out = []
    for s in range(n):
        dist = {(s, 0): 0}; heap = [(0, s, 0)]
        while heap:
            du, u, ph = heapq.heappop(heap)
            if du > dist.get((u, ph), INF):
                continue
            nxt = []
            if ph == 0:
                nxt.append((u, 1, prices[u]))
            for v, c, tc in fwd[u]:
                nxt.append((v, ph, c if ph == 0 else tc))
            for v, p2, w in nxt:
                if du + w < dist.get((v, p2), INF):
                    dist[(v, p2)] = du + w
                    heapq.heappush(heap, (du + w, v, p2))
        out.append(dist.get((s, 1), INF))
    return out

rng = random.Random(15)
for _ in range(100):
    n = rng.randint(1, 7)
    arcs = []
    for _ in range(rng.randint(0, 12)):
        if n >= 2:
            u, v = rng.sample(range(n), 2)
            arcs.append((u, v, rng.randint(1, 9), rng.randint(1, 5)))
    prices = [rng.randint(1, 40) for _ in range(n)]
    assert directed_min_cost(n, prices, arcs) == directed_layered(n, prices, arcs), (n, prices, arcs)
print("biến thể có hướng: hai Dijkstra (đồ thị đảo) = Dijkstra phân tầng")

biến thể có hướng: hai Dijkstra (đồ thị đảo) = Dijkstra phân tầng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Floyd–Warshall hai ma trận | `D` và `T` toàn cặp | `O(n³)` | `O(n²)` | kiểm chứng, `n ≲ 150` |
| 2. Dijkstra phân tầng | mô hình hai tầng, `(i,0) → (i,1)` | `O(n·(n+m) log n)` | `O(n + m)` | đối chiếu độc lập; mô hình tổng quát |
| 3. Hai Dijkstra | `d1[j] + prices[j] + d2[j]` | `O(n·m log n)` | `O(n + m)` | **lời giải nộp** |
| 4. Hai Dijkstra + cắt tỉa | dừng sớm bằng cận `2d + minP ≥ best` | `O(n·m log n)` xấu nhất, thường ít hơn | `O(n + m)` | giá/khoảng cách chênh lệch |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `dijkstra(0, build_graph(2, [[0,1,3,5]]), 2)` bằng gì? (Trọng số mang táo `3·5`.)

**Bài 2 (tính tay).** `n = 2`, `prices = [10, 4]`, `roads = [[0,1,2,3]]`. `ans` là gì? (Đi 2, mua 4, về `2·3 = 6`.)

**Bài 3 (điền TODO).** Hoàn thiện `my_combine(d1, d2, prices, i)` trả về `min(prices[i], min_j d1[j] + prices[j] + d2[j])` trên các `j` có `d1[j] < ∞`.

**Bài 4 (tự viết lại).** Viết `my_min_cost(n, prices, roads)` theo Cách 3, không nhìn code nộp.

**Bài 5 (biến thể).** Viết `min_cost_tax_on_forward(n, prices, roads)`: giả sử `tax` áp dụng cho chuyến **đi** (đi tới `j` tốn `cost·tax`) còn chuyến **về** tốn `cost`. Hãy cài bằng hai Dijkstra đổi vai trò hai trọng số.

In [23]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_combine(d1, d2, prices, i):
    # TODO: best = prices[i]; duyệt j; nếu d1[j] < INF thì best = min(best, d1[j] + prices[j] + d2[j])
    pass

# Bài 4 — TODO
def my_min_cost(n, prices, roads):
    # TODO: dựng danh sách kề (v, c, c*t); với mỗi i chạy 2 Dijkstra (trọng số 1 và 2) rồi my_combine
    pass

# Bài 5 — TODO
def min_cost_tax_on_forward(n, prices, roads):
    # TODO: tương tự Cách 3 nhưng d1 dùng cost*tax, d2 dùng cost
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: trọng số mang táo là `cost · tax`.
- Mức 2: `3 · 5 = 15`.
- Mức 3: `[0, 15]`.

**Bài 2.**
- Mức 1: tại chỗ `10`.
- Mức 2: qua cửa hàng 1: `2 + 4 + 2·3 = 12`.
- Mức 3: `ans[0] = min(10, 12) = 10`; `ans[1] = 4` (qua 0: `2 + 10 + 6 = 18`). Kết quả `[10, 4]`.

**Bài 3.**
- Mức 1: khởi tạo `best = prices[i]`.
- Mức 2: bỏ qua `j` không tới được (`d1[j] == INF`).
- Mức 3: cập nhật `best = min(best, d1[j] + prices[j] + d2[j])`.

**Bài 4.**
- Mức 1: xem lại `class Solution`.
- Mức 2: hai danh sách kề trọng số trong cùng một bộ ba.
- Mức 3: `d1 = dijkstra(i, adj, 1)`, `d2 = dijkstra(i, adj, 2)`.

**Bài 5.**
- Mức 1: đổi chỗ `1` và `2` trong hai lần gọi Dijkstra.
- Mức 2: công thức giữ nguyên: `d_forward + prices[j] + d_back`.
- Mức 3: ví dụ 1 chỉ có một cạnh nên tổng vẫn là `1·2 + 3 + 1 = 6`; hãy so với Floyd đổi vai trò hai ma trận (xem đáp án).

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert dijkstra(0, build_graph(2, [[0, 1, 3, 5]]), 2) == [0, 15]
# Bài 2
assert Solution().minCost(2, [10, 4], [[0, 1, 2, 3]]) == [10, 4] == with_floyd(2, [10, 4], [[0, 1, 2, 3]])

# Bài 3
def my_combine_ans(d1, d2, prices, i):
    best = prices[i]
    for j in range(len(prices)):
        if d1[j] < INF:
            best = min(best, d1[j] + prices[j] + d2[j])
    return best

adj = build_graph(3, [[0, 2, 1, 3], [1, 2, 3, 4], [0, 1, 5, 2]])
assert my_combine_ans(dijkstra(0, adj, 1), dijkstra(0, adj, 2), [10, 11, 1], 0) == 5
print("Bài 1–3 OK")

Bài 1–3 OK


In [25]:
# Bài 4
def my_min_cost_ans(n, prices, roads):
    adj = build_graph(n, roads)
    return [my_combine_ans(dijkstra(i, adj, 1), dijkstra(i, adj, 2), prices, i) for i in range(n)]

rng = random.Random(28)
for _ in range(100):
    args = gen_case(rng, 8, 50, 20, 10)
    assert my_min_cost_ans(*args) == with_floyd(*args)

# Bài 5
def min_cost_tax_on_forward_ans(n, prices, roads):
    adj = build_graph(n, roads)
    return [my_combine_ans(dijkstra(i, adj, 2), dijkstra(i, adj, 1), prices, i) for i in range(n)]

# kiểm chứng bằng Floyd với vai trò hai ma trận đổi chỗ
def floyd_swapped(n, prices, roads):
    D = floyd(n, roads, "taxed"); T = floyd(n, roads, "cost")
    return [min(prices[i], min([D[i][j] + prices[j] + T[j][i] for j in range(n) if D[i][j] < INF])) for i in range(n)]

for _ in range(100):
    args = gen_case(rng, 8, 50, 20, 10)
    assert min_cost_tax_on_forward_ans(*args) == floyd_swapped(*args)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (hai Dijkstra từ mỗi nguồn):**

```python
adj[u].append((v, cost, cost * tax)); adj[v].append((u, cost, cost * tax))
for i in range(n):
    d1 = dijkstra(i, trọng số = cost)           # đi tay không
    d2 = dijkstra(i, trọng số = cost * tax)     # mang táo về (đồ thị vô hướng: d2(j, i) = d2(i, j))
    ans[i] = min(prices[i], min(d1[j] + prices[j] + d2[j] for j có d1[j] < INF))
```

**Checklist trước khi submit:**

- [ ] Hai trọng số **riêng** (không gộp `cost·(1 + tax)`).
- [ ] Chỉ chuyến về mang táo mới tính `tax`.
- [ ] `best` khởi tạo `prices[i]` (mua tại chỗ); bỏ qua nút không tới được.
- [ ] Dijkstra bỏ bản ghi cũ (`du > dist[u]`).
- [ ] Đồ thị có thể **không liên thông**; `roads` có thể rỗng; `n = 1`.
- [ ] Danh sách kề thêm **cả hai chiều**.
- [ ] Đã đối chiếu Floyd–Warshall và Dijkstra phân tầng trên đồ thị nhỏ ngẫu nhiên.
- [ ] Chạy thử `n = 1000`, `m = 2000` (vài giây).